# RAW to RGB — Part 4: Camera Characterization

Even after white balance the colours are not yet correct, because the camera's spectral
sensitivities do not match the sRGB primaries. We fix this with a **3×3 colour matrix**
derived from a **colour checker chart**.

The process is:
1. Look up the **reference sRGB values** for each colour checker patch (from Babelcolor)
2. **Measure** the camera's values for the same patches
3. Find the matrix $M$ that maps measured → reference by solving a **least-squares problem**
4. Apply $M$ to the whole image

### A note on basis changes and the pseudo-inverse

If we have $n$ measured colour vectors (rows of matrix $C_{\text{meas}}$) and
want to find a $3\times3$ matrix $M$ such that $C_{\text{meas}} \cdot M \approx C_{\text{ref}}$,
the least-squares solution is:

$$M = C_{\text{meas}}^{+} \cdot C_{\text{ref}}$$

In NumPy: `M = np.linalg.lstsq(cc_meas, cc_ref, rcond=None)[0]`
which is equivalent to MATLAB's `M = ccMeas \ ccRef`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

linear2sRGB = lambda x: np.where(x > 0.0031308, 1.055 * np.clip(x, 0, None) ** (1/2.4) - 0.055, 12.92 * x)
sRGB2linear = lambda x: np.where(x > 0.04045, ((x + 0.055) / 1.055) ** 2.4, x / 12.92)

data = np.load('22_RGB_EdgeDirectedDebayer.npz')
rgb             = data['rgb']
black_noise_std = data['black_noise_std']

print(f'Loaded: rgb {rgb.shape}')

## Linear algebra recap — basis change

Before diving into colour, let's revisit how matrix inversion enables a change of basis.
This is exactly what the colour matrix $M$ does: it transforms from *camera RGB space*
to *sRGB space*.

In [ ]:
# Two basis vectors in 2D camera space
X = np.array([0.8, 0.1])
Y = np.array([0.3, 0.7])

# A point in the orthonormal (R, G) basis
P_ortho = np.array([0.6, 0.2])

# Matrix whose columns are the new basis vectors
M_xy2ortho = np.column_stack([X, Y])   # transforms FROM (X,Y) coords TO ortho
M_ortho2xy = np.linalg.inv(M_xy2ortho) # transforms FROM ortho TO (X,Y) coords

P_xy = M_ortho2xy @ P_ortho
print(f'P in orthonormal basis : {P_ortho}')
print(f'P in (X,Y) basis       : {P_xy}')
print(f'Round-trip check       : {M_xy2ortho @ P_xy}  (should equal P_ortho)')

fig, ax = plt.subplots(figsize=(5, 5))
ax.quiver(0, 0, *X,       color='b', scale=1, scale_units='xy', angles='xy', label='X')
ax.quiver(0, 0, *Y,       color='k', scale=1, scale_units='xy', angles='xy', label='Y')
ax.quiver(0, 0, *P_ortho, color='m', scale=1, scale_units='xy', angles='xy', label='P (ortho)')
ax.set_xlim(-0.2, 1.2); ax.set_ylim(-0.2, 1.2)
ax.set_aspect('equal'); ax.grid(True); ax.legend()
ax.set_xlabel('Camera Red Response')
ax.set_ylabel('Camera Green Response')
plt.tight_layout()
plt.show()

## Pseudo-inverse for overdetermined systems

With 24 colour checker patches we have 24 equations but only 9 unknowns (3×3 matrix).
NumPy's `lstsq` finds the matrix that minimises the squared colour error.

In [ ]:
# Simple example: scale factor of 2
cc_ref  = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1], [1, 1, 1]], dtype=float)
cc_meas = np.array([[2, 0, 0], [0, 2, 0], [0, 0, 2], [2, 2, 2]], dtype=float)

M_test, _, _, _ = np.linalg.lstsq(cc_meas, cc_ref, rcond=None)
print('M (should be 0.5 * Identity):')
print(np.round(M_test, 4))
print('\ncc_meas @ M (should equal cc_ref):')
print(np.round(cc_meas @ M_test, 4))

## Reference colour checker values (Babelcolor)

These are the standard sRGB values for the 24 patches of a Macbeth ColorChecker.
We linearise them (undo the sRGB gamma) to match our HDR image.

In [ ]:
cc_ref_sRGB = np.array([
    [115, 82,  68],  [195, 149, 128], [93,  123, 157], [91,  108,  65],
    [130, 129, 175], [99,  191, 171], [220, 123,  46], [72,   92, 168],
    [194,  84,  97], [91,   59, 104], [161, 189,  62], [229, 161,  40],
    [42,   63, 147], [72,  149,  72], [175,  50,  57], [238, 200,  22],
    [188,  84, 150], [0,   137, 166], [245, 245, 240], [201, 202, 201],
    [161, 162, 162], [120, 121, 121], [83,   85,  85], [50,   50,  51]
], dtype=float)

cc_ref = sRGB2linear(cc_ref_sRGB / 255.0)

def show_cc(cc):
    """Display a 24-patch colour checker as a 6×4 grid."""
    grid = cc.reshape(4, 6, 3).transpose(1, 0, 2)  # shape (6, 4, 3)
    patches = np.repeat(np.repeat(linear2sRGB(np.clip(grid, 0, 1)), 32, axis=0), 32, axis=1)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.imshow(patches)
    ax.axis('off')
    return fig

show_cc(cc_ref)
plt.title('Reference colour checker (Babelcolor)')
plt.show()

## Locate and measure colour checker patches in the image

Set `upper_left_patch_middle` and `lower_right_patch_middle` to the pixel coordinates
of the top-left and bottom-right patch centres in your image.

The script will automatically interpolate the positions of all 24 patches.

In [ ]:
# Adjust these coordinates for your image
upper_left_patch_middle  = (2617, 2414)   # (row, col)
lower_right_patch_middle = (3210, 3547)   # (row, col)

n_rows, n_cols = 4, 6  # 24 patches in a 4-row × 6-column layout

half_patch = round(
    (lower_right_patch_middle[0] - upper_left_patch_middle[0]) / (n_rows - 1) / 2 * 0.7
)

# Generate grid of patch centre coordinates
rows = np.linspace(upper_left_patch_middle[0], lower_right_patch_middle[0], n_rows)
cols = np.linspace(upper_left_patch_middle[1], lower_right_patch_middle[1], n_cols)
grid_rows, grid_cols = np.meshgrid(rows, cols)  # shape (6, 4)

# Flatten in the same order as cc_ref (row-major, rows=horizontal strips)
patch_rows = grid_rows.T.ravel().astype(int)  # shape (24,)
patch_cols = grid_cols.T.ravel().astype(int)

# Measure median value inside each patch
cc_meas = np.zeros((24, 3))
select  = np.zeros(rgb.shape[:2], dtype=float)

for i in range(24):
    r0, r1 = patch_rows[i] - half_patch, patch_rows[i] + half_patch
    c0, c1 = patch_cols[i] - half_patch, patch_cols[i] + half_patch
    patch = rgb[r0:r1, c0:c1, :]
    cc_meas[i, :] = np.median(patch.reshape(-1, 3), axis=0)
    select[r0:r1, c0:c1] = 1.0

# Visualise the sampling regions on the image
def np_binary_erosion(mask, radius=5):
    """Pure NumPy binary erosion — replaces scipy.ndimage.binary_erosion."""
    from numpy.lib.stride_tricks import sliding_window_view
    d      = 2 * radius + 1
    padded = np.pad(mask, radius, mode='constant', constant_values=False)
    return sliding_window_view(padded, (d, d)).all(axis=(-2, -1))

select_bool = select.astype(bool)
border = select_bool ^ np_binary_erosion(select_bool, radius=5)
vis = rgb.copy()
vis[border] = vis.max()

r_min = upper_left_patch_middle[0]  - 50
r_max = lower_right_patch_middle[0] + 50
c_min = upper_left_patch_middle[1]  - 50
c_max = lower_right_patch_middle[1] + 50

fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(linear2sRGB(np.clip(vis[r_min:r_max, c_min:c_max, :], 0, 1)))
ax.set_title('Colour checker with sampling regions')
ax.axis('off')
plt.tight_layout()
plt.show()

## Compute the colour matrix and apply it

In [ ]:
# Solve: cc_meas @ M ≈ cc_ref  (least squares)
M, _, _, _ = np.linalg.lstsq(cc_meas, cc_ref, rcond=None)
print('Colour matrix M:')
print(np.round(M, 4))

# Apply to the full image: reshape to (H*W, 3), multiply, reshape back
H, W, _ = rgb.shape
hdr_characterized = (rgb.reshape(-1, 3) @ M).reshape(H, W, 3)

fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(linear2sRGB(np.clip(hdr_characterized, 0, 1)))
ax.set_title('Camera-characterized image')
ax.axis('off')
plt.tight_layout()
plt.show()

## Verify with colour checker

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, cc, title in zip(axes,
                          [cc_meas, cc_ref, cc_meas @ M],
                          ['Measured', 'Target (Babelcolor)', 'After matrix']):
    show_cc(cc)
    plt.gca().set_title(title)

plt.tight_layout()
plt.show()

## Save result

In [ ]:
np.savez('23_RGB_CamCharacterized_for_sRGB.npz',
         hdr_psInv_img=hdr_characterized,
         black_noise_std=black_noise_std)

print('Saved: 23_RGB_CamCharacterized_for_sRGB.npz')